# Network Connectivity Corruption Examples

This notebook samples random English-dataset clips with `seed = 666`, applies the mobile-network corruption functions from `scripts/emulate_network_connectivity.py`, and lets you listen to the clean and corrupted audio side by side.

If the English dataset is not auto-detected on your machine, update `DATASET_ROOT` in the config cell below.

In [1]:
from __future__ import annotations

import importlib.util
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Audio, Markdown, display

SEED = 666
rng = np.random.default_rng(SEED)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd())
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "emulate_network_connectivity.py"

spec = importlib.util.spec_from_file_location("network_emulation", SCRIPT_PATH)
network_emulation = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = network_emulation
assert spec.loader is not None
spec.loader.exec_module(network_emulation)

load_mono_audio = network_emulation.load_mono_audio
save_audio = network_emulation.save_audio
apply_network_artifacts = network_emulation.apply_network_artifacts

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded emulation script: {SCRIPT_PATH}")
print(f"Seed: {SEED}")

Project root: /Users/hindy/Desktop/Academics/MIT/MIT MBAn/Spring/GenAI Lab/phronetic-ai-audioinpainting
Loaded emulation script: /Users/hindy/Desktop/Academics/MIT/MIT MBAn/Spring/GenAI Lab/phronetic-ai-audioinpainting/scripts/emulate_network_connectivity.py
Seed: 666


In [2]:
DATASET_ROOT: Path | None = None
DATASET_CANDIDATES = [
    PROJECT_ROOT / "nptel2020" / "train",
    PROJECT_ROOT / "data" / "raw" / "english",
    Path("/mnt/d/nptel2020/train"),
]
OUTPUT_DIR = PROJECT_ROOT / "tmp" / "network_connectivity_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEMO_SPECS = [
    {
        "label": "All effects | light",
        "slug": "all_light",
        "severity": "light",
        "effects": {"short_dropouts", "long_cuts", "choppy", "degradation"},
    },
    {
        "label": "All effects | medium",
        "slug": "all_medium",
        "severity": "medium",
        "effects": {"short_dropouts", "long_cuts", "choppy", "degradation"},
    },
    {
        "label": "All effects | heavy",
        "slug": "all_heavy",
        "severity": "heavy",
        "effects": {"short_dropouts", "long_cuts", "choppy", "degradation"},
    },
    {
        "label": "Short dropouts only",
        "slug": "short_dropouts_only",
        "severity": "medium",
        "effects": {"short_dropouts"},
    },
    {
        "label": "Long cuts only",
        "slug": "long_cuts_only",
        "severity": "medium",
        "effects": {"long_cuts"},
    },
    {
        "label": "Choppy congestion only",
        "slug": "choppy_only",
        "severity": "medium",
        "effects": {"choppy"},
    },
    {
        "label": "Signal degradation only",
        "slug": "degradation_only",
        "severity": "medium",
        "effects": {"degradation"},
    },
]


def resolve_dataset_root(override: Path | None, candidates: list[Path]) -> Path | None:
    if override is not None:
        override = override.expanduser()
        if override.exists():
            return override
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate.exists():
            return candidate
    return None


def collect_wav_files(root: Path | None) -> list[Path]:
    if root is None:
        return []
    return sorted(root.rglob("*.wav"))


dataset_root = resolve_dataset_root(DATASET_ROOT, DATASET_CANDIDATES)
wav_files = collect_wav_files(dataset_root)

if dataset_root is None:
    display(Markdown(
        "**English dataset not found automatically.** Set `DATASET_ROOT` to your extracted English WAV root and rerun this cell."
    ))
else:
    print(f"Dataset root: {dataset_root}")
    print(f"Detected WAV files: {len(wav_files):,}")
    if wav_files:
        display(pd.DataFrame({"example_wav": [str(path) for path in wav_files[:5]]}))

**English dataset not found automatically.** Set `DATASET_ROOT` to your extracted English WAV root and rerun this cell.

In [3]:
MAX_LISTEN_SECONDS = 12.0


def maybe_trim_audio(
    audio: np.ndarray,
    sample_rate: int,
    local_rng: np.random.Generator,
    max_seconds: float = MAX_LISTEN_SECONDS,
) -> tuple[np.ndarray, int]:
    max_samples = int(round(max_seconds * sample_rate))
    if audio.shape[0] <= max_samples:
        return audio, 0
    start = int(local_rng.integers(0, audio.shape[0] - max_samples + 1))
    end = start + max_samples
    return audio[start:end], start


EFFECT_NAME_MAP = {
    "short_dropouts": "short dropout",
    "long_cuts": "long cut",
    "choppy": "choppy window",
    "degradation": "signal degradation",
}


def summarize_events(events: list[dict]) -> pd.DataFrame:
    if not events:
        return pd.DataFrame(columns=["type", "variant", "start_sec", "end_sec"])
    frame = pd.DataFrame(events)
    preferred_columns = [
        "type",
        "variant",
        "start_sec",
        "end_sec",
        "bandlimited",
        "cutoff_hz",
        "max_attenuation_db",
    ]
    return frame[[column for column in preferred_columns if column in frame.columns]]


def build_example(example_index: int, wav_path: Path, spec: dict) -> dict:
    clean_audio, sample_rate = load_mono_audio(wav_path)
    corruption_rng = np.random.default_rng(SEED + example_index)
    listen_audio, trim_start_sample = maybe_trim_audio(clean_audio, sample_rate, corruption_rng)
    corrupted_audio, events = apply_network_artifacts(
        listen_audio,
        sample_rate,
        severity=spec["severity"],
        rng=corruption_rng,
        effects=spec["effects"],
    )

    output_wav = OUTPUT_DIR / f"{example_index:02d}_{spec['slug']}.wav"
    report_path = OUTPUT_DIR / f"{example_index:02d}_{spec['slug']}.json"
    save_audio(output_wav, corrupted_audio, sample_rate)

    report = {
        "seed": SEED,
        "example_index": example_index,
        "source_path": str(wav_path),
        "trim_start_sample": trim_start_sample,
        "trim_start_sec": round(trim_start_sample / sample_rate, 4),
        "duration_sec": round(listen_audio.shape[0] / sample_rate, 4),
        "severity": spec["severity"],
        "effects": sorted(spec["effects"]),
        "event_count": len(events),
        "events": events,
        "output_wav": str(output_wav),
    }
    report_path.write_text(json.dumps(report, indent=2))

    event_counts = Counter(event["type"] for event in events)
    return {
        "label": spec["label"],
        "slug": spec["slug"],
        "source_path": wav_path,
        "sample_rate": sample_rate,
        "clean_audio": listen_audio,
        "corrupted_audio": corrupted_audio,
        "severity": spec["severity"],
        "effects": sorted(spec["effects"]),
        "events": events,
        "event_counts": dict(event_counts),
        "events_df": summarize_events(events),
        "report_path": report_path,
        "output_wav": output_wav,
        "duration_sec": round(listen_audio.shape[0] / sample_rate, 2),
    }

In [4]:
if not wav_files:
    display(Markdown(
        "### No English WAV files found\n"
        "Update `DATASET_ROOT` to point at your extracted English dataset and rerun the config cell above."
    ))
else:
    sample_count = len(DEMO_SPECS)
    replace = len(wav_files) < sample_count
    selected_indices = rng.choice(len(wav_files), size=sample_count, replace=replace)
    selected_paths = [wav_files[int(index)] for index in selected_indices]

    examples = [
        build_example(example_index, wav_path, spec)
        for example_index, (wav_path, spec) in enumerate(zip(selected_paths, DEMO_SPECS), start=1)
    ]

    summary = pd.DataFrame(
        {
            "label": [example["label"] for example in examples],
            "severity": [example["severity"] for example in examples],
            "duration_sec": [example["duration_sec"] for example in examples],
            "event_count": [len(example["events"]) for example in examples],
            "source_path": [str(example["source_path"]) for example in examples],
            "output_wav": [str(example["output_wav"]) for example in examples],
        }
    )
    display(Markdown("## Example Summary"))
    display(summary)

    for example in examples:
        display(Markdown(f"## {example['label']}"))
        display(Markdown(
            f"Source clip: `{example['source_path']}`  \n"
            f"Saved corrupted clip: `{example['output_wav']}`  \n"
            f"Saved report: `{example['report_path']}`  \n"
            f"Severity: `{example['severity']}`  \n"
            f"Effects: `{', '.join(example['effects'])}`  \n"
            f"Events sampled: `{len(example['events'])}`"
        ))

        display(Markdown("**Original audio**"))
        display(Audio(example["clean_audio"], rate=example["sample_rate"], normalize=False))

        display(Markdown("**Corrupted audio**"))
        display(Audio(example["corrupted_audio"], rate=example["sample_rate"], normalize=False))

        if example["events_df"].empty:
            print("No events were sampled for this clip.")
        else:
            display(example["events_df"])

### No English WAV files found
Update `DATASET_ROOT` to point at your extracted English dataset and rerun the config cell above.